# NIFTY 50 Financial Risk Analysis

## Week 3 — Risk Analysis

**Student:** Pavan Poriya

---

## 1. Week 3 Objective

The objective of Week 3 is to conduct a structured financial risk analysis of the NIFTY 50 index using the historical dataset analyzed in Weeks 1 and 2.

This notebook identifies measurable risks from the data, assesses their potential impact, and proposes realistic mitigation strategies.

**Key Risk Categories Analyzed:**
- Market / Price Risk
- Volatility Risk
- Extreme Return / Shock Risk
- Downside Risk
- Drawdown Risk

**Important:** This is an educational internship project. The analysis is not personal investment advice.

## 2. Dataset Loading

We load the same NIFTY 50 historical CSV file used in Weeks 1 and 2. The dataset contains daily OHLCV (Open, High, Low, Close, Volume) data.

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import warnings
warnings.filterwarnings('ignore')

print("All libraries imported successfully.")

All libraries imported successfully.


In [2]:
df = pd.read_csv("DataSet/NIFTY50_1995_to_Feb_2026.csv")
print(f"Raw dataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
df.head()

Raw dataset shape: (4523, 10)
Columns: ['Date', 'Close', 'High', 'Low', 'Open', 'Volume', 'Ticker', 'Daily_Return_%', 'MA_50', 'MA_200']


,Date,Close,High,Low,Open,Volume,Ticker,Daily_Return_%,MA_50,MA_200
0,NaN,^NSEI,^NSEI,^NSEI,^NSEI,^NSEI,NaN,NaN,NaN,NaN
1,2007-09-17,4494.64990234375,4549.0498046875,4482.85009765625,4518.4501953125,0,^NSEI,NaN,NaN,NaN
2,2007-09-18,4546.2001953125,4551.7998046875,4481.5498046875,4494.10009765625,0,^NSEI,1.146926,NaN,NaN
3,2007-09-19,4732.35009765625,4739.0,4550.25,4550.25,0,^NSEI,4.094626,NaN,NaN
4,2007-09-20,4747.5498046875,4760.85009765625,4721.14990234375,4734.85009765625,0,^NSEI,0.321187,NaN,NaN


## 3. Data Preparation

We replicate the exact data cleaning steps from Week 1:
1. Remove the metadata row (index 0)
2. Convert `Date` to datetime
3. Convert numerical columns to appropriate types
4. Drop rows where `Close` is missing
5. Sort by date to ensure chronological order

In [3]:
# Remove metadata row (replicating Week 1 cleaning)
df = df.drop(index=0).reset_index(drop=True)

# Convert types
df["Date"] = pd.to_datetime(df["Date"])
df["Close"] = pd.to_numeric(df["Close"])
df["High"] = pd.to_numeric(df["High"])
df["Low"] = pd.to_numeric(df["Low"])
df["Open"] = pd.to_numeric(df["Open"])
df["Volume"] = pd.to_numeric(df["Volume"])

# Drop rows where Close is NaN
df = df.dropna(subset=["Close"]).reset_index(drop=True)

# Ensure sorted by date
df = df.sort_values("Date").reset_index(drop=True)

# Compute daily returns from Close prices (same as Week 1)
df["Daily_Return"] = df["Close"].pct_change() * 100

print(f"Cleaned dataset shape: {df.shape}")
print(f"Date range: {df['Date'].min().date()} to {df['Date'].max().date()}")
print(f"Total trading days: {len(df)}")

Cleaned dataset shape: (4522, 11)
Date range: 2007-09-17 to 2026-02-20
Total trading days: 4522


## 4. Market / Price Risk Analysis

Market risk refers to the potential for financial loss due to adverse movements in market prices. For the NIFTY 50 index, this manifests as significant declines in the index value.

We analyze historical price extremes, major declines, and recovery patterns to understand the magnitude of market risk present in the NIFTY 50.

In [4]:
# Market / Price Risk Statistics
print("=" * 60)
print("       MARKET / PRICE RISK ANALYSIS")
print("=" * 60)

min_close = df["Close"].min()
max_close = df["Close"].max()
price_range = max_close - min_close
current_close = df["Close"].iloc[-1]
current_date = df["Date"].iloc[-1].date()

min_close_date = df.loc[df["Close"].idxmin(), "Date"].date()
max_close_date = df.loc[df["Close"].idxmax(), "Date"].date()

print(f"Minimum Close:            ₹{min_close:,.2f} on {min_close_date}")
print(f"Maximum Close:            ₹{max_close:,.2f} on {max_close_date}")
print(f"Price Range:              ₹{price_range:,.2f}")
print(f"Latest Close:             ₹{current_close:,.2f} on {current_date}")
print(f"Total Appreciation:       {((max_close / min_close) - 1) * 100:,.2f}%")
print("=" * 60)

       MARKET / PRICE RISK ANALYSIS
Minimum Close:            ₹2,524.20 on 2008-10-27
Maximum Close:            ₹26,328.55 on 2026-01-02
Price Range:              ₹23,804.35
Latest Close:             ₹25,571.25 on 2026-02-20
Total Appreciation:       943.05%


In [5]:
# Identify major price declines (rolling 20-day peak-to-trough)
df["Cumulative_Max"] = df["Close"].cummax()
df["Drawdown"] = (df["Close"] - df["Cumulative_Max"]) / df["Cumulative_Max"] * 100

# Find top 5 worst drawdown periods
worst_drawdown_idx = df["Drawdown"].idxmin()
worst_drawdown_date = df.loc[worst_drawdown_idx, "Date"].date()
worst_drawdown_pct = df["Drawdown"].min()
worst_drawdown_peak = df.loc[worst_drawdown_idx, "Cumulative_Max"]
worst_drawdown_trough = df.loc[worst_drawdown_idx, "Close"]

print(f"Maximum Drawdown:         {worst_drawdown_pct:.2f}%")
print(f"Drawdown Peak:            ₹{worst_drawdown_peak:,.2f}")
print(f"Drawdown Trough:          ₹{worst_drawdown_trough:,.2f}")
print(f"Drawdown Date:            {worst_drawdown_date}")
print(f"Absolute Loss:            ₹{worst_drawdown_peak - worst_drawdown_trough:,.2f}")

Maximum Drawdown:         -59.86%
Drawdown Peak:            ₹6,287.85
Drawdown Trough:          ₹2,524.20
Drawdown Date:            2008-10-27
Absolute Loss:            ₹3,763.65


In [6]:
# Major single-day declines
daily_ret = df["Daily_Return"].dropna()

# Top 5 worst single-day losses
worst_days = df.nsmallest(5, "Daily_Return")[["Date", "Close", "Daily_Return"]]
print("Top 5 Worst Single-Day Losses:")
print(worst_days.to_string(index=False))
print()

# Top 5 best single-day gains
best_days = df.nlargest(5, "Daily_Return")[["Date", "Close", "Daily_Return"]]
print("Top 5 Best Single-Day Gains:")
print(best_days.to_string(index=False))

Top 5 Worst Single-Day Losses:
      Date       Close  Daily_Return
2020-03-23 7610.250000    -12.980466
2008-10-24 2584.000000    -12.202909
2008-01-21 5208.799805     -8.702435
2020-03-12 9590.150391     -8.301939
2020-03-16 9197.400391     -7.612100

Top 5 Best Single-Day Gains:
      Date       Close  Daily_Return
2009-05-18 4323.149902     17.744066
2020-04-07 8792.200195      8.763210
2008-10-31 2885.600098      6.990973
2008-01-25 5383.350098      6.951492
2008-10-29 2697.050049      6.847718


In [7]:
# Market risk visualization: Historical Close price with major drawdown highlighted
fig, axes = plt.subplots(2, 1, figsize=(14, 8), gridspec_kw={'height_ratios': [2, 1]})

# Plot 1: Historical Close Price
axes[0].plot(df["Date"], df["Close"], color="#1f77b4", linewidth=1.2, label="NIFTY 50 Close")
axes[0].axhline(y=min_close, color="red", linestyle="--", alpha=0.6, label=f"Minimum: ₹{min_close:,.0f}")
axes[0].axhline(y=max_close, color="green", linestyle="--", alpha=0.6, label=f"Maximum: ₹{max_close:,.0f}")
axes[0].set_title("NIFTY 50 — Historical Closing Price with Price Extremes", fontsize=13, fontweight="bold")
axes[0].set_ylabel("Closing Price (₹)", fontsize=11)
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# Plot 2: Daily Returns
axes[1].bar(df["Date"], df["Daily_Return"], color=np.where(df["Daily_Return"] >= 0, "green", "red"), alpha=0.6, width=1)
axes[1].axhline(y=0, color="black", linewidth=0.5)
axes[1].set_title("Daily Returns (%)", fontsize=11)
axes[1].set_ylabel("Return (%)", fontsize=10)
axes[1].set_xlabel("Date", fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("assets/week3_market_risk.png", dpi=150, bbox_inches="tight")
plt.show()
print("Chart saved: assets/week3_market_risk.png")

Chart saved: assets/week3_market_risk.png


## 5. Volatility Risk Analysis

Volatility risk refers to the uncertainty and magnitude of price fluctuations over time. Higher volatility means greater uncertainty and potentially larger losses (or gains) in short periods.

We use rolling standard deviation of daily returns to measure time-varying volatility.

In [8]:
# Rolling volatility (30-day standard deviation of daily returns)
df["Volatility_30d"] = df["Daily_Return"].rolling(window=30).std()

# Overall volatility statistics
vol_30d = df["Volatility_30d"].dropna()

avg_vol = vol_30d.mean()
max_vol = vol_30d.max()
min_vol = vol_30d.min()
max_vol_date = df.loc[vol_30d.idxmax(), "Date"].date()
min_vol_date = df.loc[vol_30d.idxmin(), "Date"].date()

print("=" * 60)
print("       VOLATILITY RISK ANALYSIS (30-Day Rolling)")
print("=" * 60)
print(f"Average 30-day Volatility:  {avg_vol:.4f}%")
print(f"Maximum 30-day Volatility:  {max_vol:.4f}% on {max_vol_date}")
print(f"Minimum 30-day Volatility:  {min_vol:.4f}% on {min_vol_date}")
print(f"Volatility Range:           {max_vol - min_vol:.4f}%")
print(f"Median 30-day Volatility:   {vol_30d.median():.4f}%")
print(f"Std Dev of Volatility:      {vol_30d.std():.4f}%")
print("=" * 60)

       VOLATILITY RISK ANALYSIS (30-Day Rolling)
Average 30-day Volatility:  1.1028%
Maximum 30-day Volatility:  4.8409% on 2008-11-24
Minimum 30-day Volatility:  0.3897% on 2017-07-07
Volatility Range:           4.4512%
Median 30-day Volatility:   0.9080%
Std Dev of Volatility:      0.6847%


In [9]:
# Yearly average volatility
df_valid = df.dropna(subset=["Volatility_30d"]).copy()
df_valid["Year"] = df_valid["Date"].dt.year
yearly_vol = df_valid.groupby("Year")["Volatility_30d"].mean().sort_values(ascending=False)

print("Top 5 Most Volatile Years (by average 30-day rolling volatility):")
for year, vol in yearly_vol.head(5).items():
    print(f"  {year}: {vol:.4f}%")
print()
print("Top 5 Least Volatile Years:")
for year, vol in yearly_vol.tail(5).items():
    print(f"  {year}: {vol:.4f}%")

Top 5 Most Volatile Years (by average 30-day rolling volatility):
  2008: 2.6083%
  2009: 2.1171%
  2007: 2.0244%
  2020: 1.5720%
  2011: 1.2865%

Top 5 Least Volatile Years:
  2018: 0.7558%
  2025: 0.7183%
  2026: 0.6082%
  2023: 0.6068%
  2017: 0.5842%


In [10]:
# High-volatility threshold analysis
high_vol_threshold = avg_vol + 2 * vol_30d.std()
high_vol_days = df_valid[df_valid["Volatility_30d"] > high_vol_threshold]

print(f"High-Volatility Threshold (> mean + 2σ): {high_vol_threshold:.4f}%")
print(f"Number of high-volatility days: {len(high_vol_days)}")
print(f"Percentage of observed days: {len(high_vol_days) / len(df_valid) * 100:.2f}%")

if len(high_vol_days) > 0:
    print(f"\nHigh-volatility periods:")
    for _, row in high_vol_days[["Date", "Volatility_30d"]].iterrows():
        print(f"  {row['Date'].date()}: {row['Volatility_30d']:.4f}%")

High-Volatility Threshold (> mean + 2σ): 2.4721%
Number of high-volatility days: 227
Percentage of observed days: 5.05%

High-volatility periods:
  2008-01-23: 2.7560%
  2008-01-24: 2.7546%
  2008-01-25: 3.0697%
  2008-01-28: 3.0759%
  2008-01-29: 3.0772%
  2008-01-30: 2.9993%
  2008-01-31: 2.9993%
  2008-02-01: 3.0789%
  2008-02-04: 3.1253%
  2008-02-05: 3.0385%
  2008-02-06: 3.0601%
  2008-02-07: 3.1116%
  2008-02-08: 3.1107%
  2008-02-11: 3.2082%
  2008-02-12: 3.2052%
  2008-02-13: 3.2326%
  2008-02-14: 3.4258%
  2008-02-15: 3.4346%
  2008-02-18: 3.4329%
  2008-02-19: 3.4325%
  2008-02-20: 3.4488%
  2008-02-21: 3.4488%
  2008-02-22: 3.4461%
  2008-02-25: 3.4707%
  2008-02-26: 3.4731%
  2008-02-27: 3.4557%
  2008-02-28: 3.4577%
  2008-02-29: 3.4090%
  2008-03-03: 3.1588%
  2008-03-04: 2.9809%
  2008-03-05: 2.7536%
  2008-03-07: 2.7452%
  2008-03-13: 2.5386%
  2008-03-14: 2.5955%
  2008-03-17: 2.6446%
  2008-03-18: 2.5823%
  2008-03-19: 2.5909%
  2008-03-24: 2.5625%
  2008-03-25: 2.73

In [11]:
# Volatility visualization
fig, ax = plt.subplots(figsize=(14, 5))

ax.plot(df["Date"], df["Volatility_30d"], color="#d62728", linewidth=1, label="30-Day Rolling Volatility")
ax.axhline(y=avg_vol, color="blue", linestyle="--", alpha=0.7, label=f"Average: {avg_vol:.2f}%")
ax.axhline(y=high_vol_threshold, color="orange", linestyle="--", alpha=0.7, label=f"High-Vol Threshold: {high_vol_threshold:.2f}%")
ax.fill_between(df["Date"], 0, df["Volatility_30d"], alpha=0.15, color="red")

ax.set_title("NIFTY 50 — 30-Day Rolling Volatility of Daily Returns", fontsize=13, fontweight="bold")
ax.set_xlabel("Date", fontsize=11)
ax.set_ylabel("Volatility (%)", fontsize=11)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("assets/week3_volatility.png", dpi=150, bbox_inches="tight")
plt.show()
print("Chart saved: assets/week3_volatility.png")

Chart saved: assets/week3_volatility.png


## 6. Extreme Return / Shock Risk

Extreme return risk refers to the possibility of unusually large positive or negative price movements in a single trading day. These "shock" events can cause significant portfolio losses and are a key component of financial risk.

We analyze the frequency, magnitude, and timing of extreme daily returns.

In [12]:
# Extreme return analysis
max_gain = daily_ret.max()
max_loss = daily_ret.min()
max_gain_date = df.loc[df["Daily_Return"].idxmax(), "Date"].date()
max_loss_date = df.loc[df["Daily_Return"].idxmin(), "Date"].date()

# Threshold: ±2% for large moves (standard threshold)
threshold = 2.0
large_positive = df[df["Daily_Return"] > threshold]
large_negative = df[df["Daily_Return"] < -threshold]

print("=" * 60)
print("       EXTREME RETURN / SHOCK RISK ANALYSIS")
print("=" * 60)
print(f"Maximum Daily Gain:        +{max_gain:.2f}% on {max_gain_date}")
print(f"Maximum Daily Loss:        {max_loss:.2f}% on {max_loss_date}")
print(f"\nThreshold for large moves: ±{threshold}%")
print(f"Number of large positive moves (>{threshold}%):  {len(large_positive)}")
print(f"Number of large negative moves (<-{threshold}%): {len(large_negative)}")
print(f"Total large moves:         {len(large_positive) + len(large_negative)}")
print(f"Percentage of trading days: {(len(large_positive) + len(large_negative)) / len(daily_ret.dropna()) * 100:.2f}%")

if len(large_positive) > 0:
    print(f"\nAverage magnitude of large positive moves: +{large_positive['Daily_Return'].mean():.2f}%")
if len(large_negative) > 0:
    print(f"Average magnitude of large negative moves:  {large_negative['Daily_Return'].mean():.2f}%")

print(f"\nMean daily return:         {daily_ret.mean():.4f}%")
print(f"Std dev of daily returns:  {daily_ret.std():.4f}%")
print(f"Skewness:                  {daily_ret.skew():.4f}")
print(f"Kurtosis:                  {daily_ret.kurtosis():.4f}")
print("=" * 60)

       EXTREME RETURN / SHOCK RISK ANALYSIS
Maximum Daily Gain:        +17.74% on 2009-05-18
Maximum Daily Loss:        -12.98% on 2020-03-23

Threshold for large moves: ±2.0%
Number of large positive moves (>2.0%):  193
Number of large negative moves (<-2.0%): 194
Total large moves:         387
Percentage of trading days: 8.56%

Average magnitude of large positive moves: +3.21%
Average magnitude of large negative moves:  -3.23%

Mean daily return:         0.0469%
Std dev of daily returns:  1.3015%
Skewness:                  0.0561
Kurtosis:                  16.0672


In [13]:
# Extreme return visualization: Distribution of daily returns with extreme thresholds
fig, ax = plt.subplots(figsize=(12, 5))

n, bins, patches = ax.hist(daily_ret.dropna(), bins=80, color="#1f77b4", alpha=0.7, edgecolor="white")

# Color extreme bins
for patch, left_edge in zip(patches, bins[:-1]):
    if left_edge > threshold:
        patch.set_facecolor("green")
    elif left_edge < -threshold:
        patch.set_facecolor("red")

ax.axvline(x=threshold, color="green", linestyle="--", linewidth=1.5, label=f"+{threshold}% threshold")
ax.axvline(x=-threshold, color="red", linestyle="--", linewidth=1.5, label=f"-{threshold}% threshold")
ax.axvline(x=0, color="black", linewidth=0.5)

ax.set_title("Distribution of Daily Returns with Extreme Move Thresholds", fontsize=13, fontweight="bold")
ax.set_xlabel("Daily Return (%)", fontsize=11)
ax.set_ylabel("Frequency", fontsize=11)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("assets/week3_extreme_returns.png", dpi=150, bbox_inches="tight")
plt.show()
print("Chart saved: assets/week3_extreme_returns.png")

Chart saved: assets/week3_extreme_returns.png


In [14]:
# Identify major shock periods
print("Major Shock Periods (clusters of extreme returns):")
print()

# Look for periods with multiple extreme returns within 10 days
extreme_days = df[df["Daily_Return"].abs() > threshold].copy()
extreme_days["Year"] = extreme_days["Date"].dt.year
yearly_extremes = extreme_days.groupby("Year").agg(
    Count=("Daily_Return", "count"),
    Avg_Magnitude=("Daily_Return", lambda x: x.abs().mean()),
    Worst_Drop=("Daily_Return", "min"),
    Best_Gain=("Daily_Return", "max")
).sort_values("Count", ascending=False)

print(yearly_extremes.to_string())

Major Shock Periods (clusters of extreme returns):

      Count  Avg_Magnitude  Worst_Drop  Best_Gain
Year                                             
2008    104       3.766821  -12.202909   6.990973
2009     65       3.364940   -6.180935  17.744066
2020     42       3.797198  -12.980466   8.763210
2011     31       2.602106   -4.083185   3.618524
2013     20       2.695524   -4.082854   3.808712
2007     18       3.407271   -4.476085   5.588353
2022     17       2.644180   -4.778104   3.025913
2010     14       2.500549   -3.091112   3.498377
2012     14       2.441065   -2.727785   3.049110
2015     13       2.607356   -5.915098   2.616723
2016     11       2.464870   -3.317074   3.366943
2021      9       2.917443   -3.763569   4.742351
2024      8       2.984335   -5.929360   3.362424
2019      6       3.032462   -2.138240   5.319113
2018      6       2.343501   -2.668111   2.323964
2014      4       2.361981   -2.105388   2.987300
2025      3       3.083934   -3.243255   3.81830

## 7. Downside Risk Analysis

Downside risk focuses specifically on the probability and magnitude of negative returns. This is directly relevant to investors because losses typically have a greater psychological and financial impact than equivalent gains.

We analyze the frequency, severity, and distribution of negative trading days.

In [15]:
# Downside risk statistics
valid_returns = daily_ret.dropna()
negative_returns = valid_returns[valid_returns < 0]
positive_returns = valid_returns[valid_returns >= 0]

# Basic downside stats
total_days = len(valid_returns)
neg_days = len(negative_returns)
pos_days = len(positive_returns)

print("=" * 60)
print("       DOWNSIDE RISK ANALYSIS")
print("=" * 60)
print(f"Total trading days with returns: {total_days}")
print(f"Negative trading days:           {neg_days} ({neg_days/total_days*100:.2f}%)")
print(f"Positive trading days:           {pos_days} ({pos_days/total_days*100:.2f}%)")
print()
print(f"Average negative return:         {negative_returns.mean():.4f}%")
print(f"Average positive return:         {positive_returns.mean():.4f}%")
print(f"Worst single-day loss:           {negative_returns.min():.2f}%")
print(f"Worst 5th percentile loss:       {np.percentile(negative_returns, 5):.2f}%")
print(f"Worst 1st percentile loss:       {np.percentile(negative_returns, 1):.2f}%")
print()

# Value at Risk (VaR) - Historical method
var_95 = np.percentile(valid_returns, 5)
var_99 = np.percentile(valid_returns, 1)

print(f"Historical Value at Risk (VaR):")
print(f"  VaR at 95% confidence:  {var_95:.2f}% (daily)")
print(f"  VaR at 99% confidence:  {var_99:.2f}% (daily)")
print()

# Conditional VaR (Expected Shortfall)
cvar_95 = valid_returns[valid_returns <= var_95].mean()
cvar_99 = valid_returns[valid_returns <= var_99].mean()

print(f"Conditional VaR (Expected Shortfall):")
print(f"  CVaR at 95% confidence: {cvar_95:.2f}% (daily)")
print(f"  CVaR at 99% confidence: {cvar_99:.2f}% (daily)")
print("=" * 60)

       DOWNSIDE RISK ANALYSIS
Total trading days with returns: 4521
Negative trading days:           2118 (46.85%)
Positive trading days:           2403 (53.15%)

Average negative return:         -0.8674%
Average positive return:         0.8528%
Worst single-day loss:           -12.98%
Worst 5th percentile loss:       -2.61%
Worst 1st percentile loss:       -5.06%

Historical Value at Risk (VaR):
  VaR at 95% confidence:  -1.83% (daily)
  VaR at 99% confidence:  -3.57% (daily)

Conditional VaR (Expected Shortfall):
  CVaR at 95% confidence: -3.04% (daily)
  CVaR at 99% confidence: -5.33% (daily)


In [16]:
# Downside risk visualization: Distribution of negative returns
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Negative returns distribution
axes[0].hist(negative_returns, bins=50, color="red", alpha=0.7, edgecolor="white")
axes[0].axvline(x=var_95, color="orange", linestyle="--", linewidth=1.5, label=f"VaR 95%: {var_95:.2f}%")
axes[0].axvline(x=var_99, color="darkred", linestyle="--", linewidth=1.5, label=f"VaR 99%: {var_99:.2f}%")
axes[0].set_title("Distribution of Negative Daily Returns", fontsize=12, fontweight="bold")
axes[0].set_xlabel("Daily Return (%)", fontsize=10)
axes[0].set_ylabel("Frequency", fontsize=10)
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

# Cumulative downside impact
axes[1].plot(valid_returns.index, valid_returns.cumsum(), color="#1f77b4", linewidth=1)
axes[1].fill_between(valid_returns.index, valid_returns.cumsum(), 0, 
                     where=valid_returns.cumsum() < 0, color="red", alpha=0.3)
axes[1].set_title("Cumulative Return Over Time", fontsize=12, fontweight="bold")
axes[1].set_xlabel("Trading Day Index", fontsize=10)
axes[1].set_ylabel("Cumulative Return (%)", fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("assets/week3_downside_risk.png", dpi=150, bbox_inches="tight")
plt.show()
print("Chart saved: assets/week3_downside_risk.png")

Chart saved: assets/week3_downside_risk.png


## 8. Drawdown Analysis

Drawdown measures the decline from a historical peak in the index value. It represents the maximum loss an investor would have experienced from the peak to the subsequent trough.

Drawdown analysis is critical for understanding the worst-case scenario for investors who held the index over the analyzed period.

In [17]:
# Drawdown analysis
# Running maximum and drawdown (already computed above, but let's formalize)
df["Running_Max"] = df["Close"].cummax()
df["Drawdown_Pct"] = (df["Close"] - df["Running_Max"]) / df["Running_Max"] * 100
df["Drawdown_Abs"] = df["Close"] - df["Running_Max"]

# Maximum drawdown
max_dd_idx = df["Drawdown_Pct"].idxmin()
max_dd_date = df.loc[max_dd_idx, "Date"].date()
max_dd_pct = df["Drawdown_Pct"].min()
max_dd_peak = df.loc[max_dd_idx, "Running_Max"]
max_dd_trough = df.loc[max_dd_idx, "Close"]
max_dd_abs = df.loc[max_dd_idx, "Drawdown_Abs"]

# Find when the peak occurred before max drawdown
peak_idx = df.loc[:max_dd_idx, "Close"].idxmax()
peak_date = df.loc[peak_idx, "Date"].date()

# Find recovery date (first time Close exceeds the peak after the trough)
recovery_df = df.loc[max_dd_idx:]
recovery_candidates = recovery_df[recovery_df["Close"] >= max_dd_peak]
if len(recovery_candidates) > 0:
    recovery_date = recovery_candidates.iloc[0]["Date"].date()
    recovery_days = (pd.Timestamp(recovery_date) - pd.Timestamp(max_dd_date)).days
else:
    recovery_date = "Not yet recovered"
    recovery_days = "N/A"

print("=" * 60)
print("       DRAWDOWN ANALYSIS")
print("=" * 60)
print(f"Maximum Drawdown:          {max_dd_pct:.2f}%")
print(f"Absolute Loss:             ₹{abs(max_dd_abs):,.2f}")
print(f"Peak Price:                ₹{max_dd_peak:,.2f} on {peak_date}")
print(f"Trough Price:              ₹{max_dd_trough:,.2f} on {max_dd_date}")
print(f"Recovery Date:             {recovery_date}")
print(f"Recovery Time:             {recovery_days} days" if isinstance(recovery_days, int) else f"Recovery Time: {recovery_days}")

# Top 5 drawdown periods
print(f"\nTop 5 Drawdown Periods:")
# Find drawdown troughs (local minima of drawdown)
drawdown_troughs = []
in_drawdown = False
for i in range(1, len(df)):
    if df.loc[i, "Drawdown_Pct"] < -2 and not in_drawdown:
        in_drawdown = True
        start_idx = i
    elif (df.loc[i, "Drawdown_Pct"] >= 0 or i == len(df) - 1) and in_drawdown:
        in_drawdown = False
        trough_local = df.loc[start_idx:i, "Drawdown_Pct"].min()
        trough_local_idx = df.loc[start_idx:i, "Drawdown_Pct"].idxmin()
        drawdown_troughs.append({
            "Start": df.loc[start_idx, "Date"].date(),
            "Trough Date": df.loc[trough_local_idx, "Date"].date(),
            "Max Drawdown": trough_local
        })

drawdown_troughs.sort(key=lambda x: x["Max Drawdown"])
for i, dd in enumerate(drawdown_troughs[:5], 1):
    print(f"  {i}. {dd['Max Drawdown']:.2f}% (Trough: {dd['Trough Date']})")

print("=" * 60)

       DRAWDOWN ANALYSIS
Maximum Drawdown:          -59.86%
Absolute Loss:             ₹3,763.65
Peak Price:                ₹6,287.85 on 2008-01-08
Trough Price:              ₹2,524.20 on 2008-10-27
Recovery Date:             2010-11-09
Recovery Time:             743 days

Top 5 Drawdown Periods:


  1. -59.86% (Trough: 2008-10-27)
  2. -38.44% (Trough: 2020-03-23)
  3. -27.89% (Trough: 2011-12-20)
  4. -22.52% (Trough: 2016-02-25)
  5. -17.23% (Trough: 2022-06-17)


In [18]:
# Drawdown visualization
fig, axes = plt.subplots(2, 1, figsize=(14, 8), gridspec_kw={'height_ratios': [1, 1]})

# Plot 1: Historical Close with Running Maximum
axes[0].plot(df["Date"], df["Close"], color="#1f77b4", linewidth=1.2, label="NIFTY 50 Close")
axes[0].plot(df["Date"], df["Running_Max"], color="green", linewidth=1, alpha=0.7, linestyle="--", label="Running Maximum")
axes[0].fill_between(df["Date"], df["Close"], df["Running_Max"], 
                     where=df["Close"] < df["Running_Max"], color="red", alpha=0.2, label="Drawdown Region")
axes[0].set_title("NIFTY 50 — Historical Price with Running Maximum and Drawdown", fontsize=13, fontweight="bold")
axes[0].set_ylabel("Price (₹)", fontsize=11)
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# Plot 2: Drawdown Percentage
axes[1].fill_between(df["Date"], df["Drawdown_Pct"], 0, color="red", alpha=0.4)
axes[1].plot(df["Date"], df["Drawdown_Pct"], color="darkred", linewidth=0.8)
axes[1].axhline(y=-10, color="orange", linestyle="--", alpha=0.7, label="-10% threshold")
axes[1].axhline(y=-20, color="red", linestyle="--", alpha=0.7, label="-20% threshold")
axes[1].set_title("NIFTY 50 — Drawdown from Running Maximum", fontsize=13, fontweight="bold")
axes[1].set_xlabel("Date", fontsize=11)
axes[1].set_ylabel("Drawdown (%)", fontsize=11)
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("assets/week3_drawdown.png", dpi=150, bbox_inches="tight")
plt.show()
print("Chart saved: assets/week3_drawdown.png")

Chart saved: assets/week3_drawdown.png


## 9. Risk Assessment

We now compile all risk metrics into a comprehensive assessment. Each risk is evaluated based on its evidence from the data, potential impact, and severity.

In [19]:
# Comprehensive risk assessment summary
print("=" * 70)
print("       COMPREHENSIVE RISK ASSESSMENT SUMMARY")
print("=" * 70)

print(f"\n1. MARKET / PRICE RISK")
print(f"   Price Range: ₹{min_close:,.0f} to ₹{max_close:,.0f}")
print(f"   Maximum Drawdown: {max_dd_pct:.2f}%")
print(f"   Absolute Drawdown: ₹{abs(max_dd_abs):,.0f}")
print(f"   Severity: VERY HIGH")

print(f"\n2. VOLATILITY RISK")
print(f"   Average 30-day Volatility: {avg_vol:.2f}%")
print(f"   Maximum 30-day Volatility: {max_vol:.2f}% on {max_vol_date}")
print(f"   High-Vol Days: {len(high_vol_days)} ({len(high_vol_days)/len(df_valid)*100:.1f}%)")
print(f"   Severity: HIGH")

print(f"\n3. EXTREME RETURN / SHOCK RISK")
print(f"   Worst Single-Day Loss: {max_loss:.2f}% on {max_loss_date}")
print(f"   Best Single-Day Gain: +{max_gain:.2f}% on {max_gain_date}")
print(f"   Days with ±{threshold}% moves: {len(large_positive) + len(large_negative)} ({(len(large_positive) + len(large_negative))/total_days*100:.2f}%)")
print(f"   Kurtosis: {daily_ret.kurtosis():.2f} (leptokurtic = fat tails)")
print(f"   Severity: HIGH")

print(f"\n4. DOWNSIDE RISK")
print(f"   Negative Trading Days: {neg_days} ({neg_days/total_days*100:.1f}%)")
print(f"   VaR (95%): {var_95:.2f}%")
print(f"   VaR (99%): {var_99:.2f}%")
print(f"   CVaR (95%): {cvar_95:.2f}%")
print(f"   Severity: MODERATE")

print(f"\n5. DRAWDOWN RISK")
print(f"   Maximum Drawdown: {max_dd_pct:.2f}%")
print(f"   Peak-to-Trough Loss: ₹{abs(max_dd_abs):,.0f}")
print(f"   Recovery: {'Achieved' if isinstance(recovery_days, int) else 'Not yet recovered'}")
if isinstance(recovery_days, int):
    print(f"   Recovery Time: {recovery_days} days ({recovery_days/365:.1f} years)")
print(f"   Severity: VERY HIGH")
print("=" * 70)

       COMPREHENSIVE RISK ASSESSMENT SUMMARY

1. MARKET / PRICE RISK
   Price Range: ₹2,524 to ₹26,329
   Maximum Drawdown: -59.86%
   Absolute Drawdown: ₹3,764
   Severity: VERY HIGH

2. VOLATILITY RISK
   Average 30-day Volatility: 1.10%
   Maximum 30-day Volatility: 4.84% on 2008-11-24
   High-Vol Days: 227 (5.1%)
   Severity: HIGH

3. EXTREME RETURN / SHOCK RISK
   Worst Single-Day Loss: -12.98% on 2020-03-23
   Best Single-Day Gain: +17.74% on 2009-05-18
   Days with ±2.0% moves: 387 (8.56%)
   Kurtosis: 16.07 (leptokurtic = fat tails)
   Severity: HIGH

4. DOWNSIDE RISK
   Negative Trading Days: 2118 (46.8%)
   VaR (95%): -1.83%
   VaR (99%): -3.57%
   CVaR (95%): -3.04%
   Severity: MODERATE

5. DRAWDOWN RISK
   Maximum Drawdown: -59.86%
   Peak-to-Trough Loss: ₹3,764
   Recovery: Achieved
   Recovery Time: 743 days (2.0 years)
   Severity: VERY HIGH


## 10. Risk Ranking

Based on the quantitative analysis above, we rank the identified risks from most to least severe.

In [20]:
# Risk ranking table
risk_data = {
    "Risk Category": [
        "Market / Price Risk",
        "Drawdown Risk",
        "Volatility Risk",
        "Extreme Return / Shock Risk",
        "Downside Risk"
    ],
    "Key Evidence": [
        f"Price range ₹{min_close:,.0f}–₹{max_close:,.0f}; Max drawdown {max_dd_pct:.2f}%",
        f"Max drawdown {max_dd_pct:.2f}% ({abs(max_dd_abs):,.0f} loss); Recovery: {'Yes' if isinstance(recovery_days, int) else 'No'}",
        f"Avg vol {avg_vol:.2f}%; Max vol {max_vol:.2f}%; {len(high_vol_days)} high-vol days",
        f"Worst day {max_loss:.2f}%; {len(large_positive)+len(large_negative)} extreme days; Kurtosis {daily_ret.kurtosis():.1f}",
        f"{neg_days} neg days ({neg_days/total_days*100:.1f}%); VaR95 {var_95:.2f}%; VaR99 {var_99:.2f}%"
    ],
    "Potential Impact": [
        "Large capital losses during market downturns; extended recovery periods",
        "Prolonged portfolio value decline; psychological and financial stress",
        "Increased uncertainty; larger-than-expected daily moves",
        "Sudden large losses in single sessions; portfolio value shocks",
        "Consistent negative returns eroding portfolio value over time"
    ],
    "Severity": [
        "Very High",
        "Very High",
        "High",
        "High",
        "Moderate"
    ],
    "Mitigation Strategy": [
        "Diversification across asset classes; position sizing; exposure limits",
        "Maximum drawdown limits; stop-loss mechanisms; periodic rebalancing",
        "Volatility monitoring; reduce exposure during high-vol periods; risk budgets",
        "Stress testing; scenario analysis; liquidity buffers; hedging",
        "Risk monitoring dashboards; diversification; dollar-cost averaging"
    ]
}

risk_df = pd.DataFrame(risk_data)
print("RISK RANKING TABLE")
print("=" * 70)
for i, row in risk_df.iterrows():
    print(f"\n{i+1}. {row['Risk Category']} — Severity: {row['Severity']}")
    print(f"   Evidence: {row['Key Evidence']}")
    print(f"   Impact: {row['Potential Impact']}")
    print(f"   Mitigation: {row['Mitigation Strategy']}")

RISK RANKING TABLE

1. Market / Price Risk — Severity: Very High
   Evidence: Price range ₹2,524–₹26,329; Max drawdown -59.86%
   Impact: Large capital losses during market downturns; extended recovery periods
   Mitigation: Diversification across asset classes; position sizing; exposure limits

2. Drawdown Risk — Severity: Very High
   Evidence: Max drawdown -59.86% (3,764 loss); Recovery: Yes
   Impact: Prolonged portfolio value decline; psychological and financial stress
   Mitigation: Maximum drawdown limits; stop-loss mechanisms; periodic rebalancing

3. Volatility Risk — Severity: High
   Evidence: Avg vol 1.10%; Max vol 4.84%; 227 high-vol days
   Impact: Increased uncertainty; larger-than-expected daily moves
   Mitigation: Volatility monitoring; reduce exposure during high-vol periods; risk budgets

4. Extreme Return / Shock Risk — Severity: High
   Evidence: Worst day -12.98%; 387 extreme days; Kurtosis 16.1
   Impact: Sudden large losses in single sessions; portfolio value s

## 11. Mitigation Strategies

The following mitigation strategies are presented as general risk-management approaches. They are not personalized investment advice.

In [21]:
# Mitigation strategies summary
print("=" * 70)
print("       RISK MITIGATION STRATEGIES")
print("=" * 70)

strategies = {
    "Market / Price Risk": [
        "Diversification: Spread investments across asset classes (equity, debt, gold) to reduce single-market exposure.",
        "Position Sizing: Limit the proportion of portfolio allocated to any single index or sector.",
        "Exposure Limits: Set maximum allocation thresholds for equity exposure based on risk tolerance.",
        "Long-Term Horizon: Maintain a long investment horizon to ride through short-term market declines."
    ],
    "Volatility Risk": [
        "Volatility Monitoring: Track rolling volatility metrics regularly to identify unusual market conditions.",
        "Risk Budgets: Allocate risk budgets across portfolio components; reduce exposure when volatility exceeds thresholds.",
        "Adaptive Allocation: Consider adjusting portfolio allocation during periods of elevated volatility.",
        "Options Hedging: Use protective strategies to limit downside during volatile periods (if available)."
    ],
    "Extreme Return / Shock Risk": [
        "Stress Testing: Regularly test portfolio performance under extreme historical scenarios.",
        "Scenario Analysis: Model the impact of hypothetical extreme events on portfolio value.",
        "Liquidity Buffers: Maintain adequate liquid reserves to meet obligations during market shocks.",
        "Circuit Breaker Policies: Implement automatic risk-reduction triggers when losses exceed thresholds."
    ],
    "Downside Risk": [
        "Downside Monitoring: Track downside-specific metrics (VaR, CVaR) alongside standard volatility.",
        "Stop-Loss Mechanisms: Implement systematic rules for reducing exposure after significant declines.",
        "Diversification: Use assets with low or negative correlation to reduce portfolio downside.",
        "Risk Parity: Weight portfolio positions by their risk contribution rather than capital allocation."
    ],
    "Drawdown Risk": [
        "Maximum Drawdown Limits: Set acceptable maximum drawdown thresholds; take action when breached.",
        "Periodic Rebalancing: Regularly rebalance portfolio to maintain target risk levels.",
        "Drawdown Recovery Planning: Prepare action plans for different drawdown severity levels.",
        "Historical Worst-Case Preparedness: Use maximum drawdown data to stress-test portfolio resilience."
    ]
}

for risk, strats in strategies.items():
    print(f"\n{risk}:")
    for s in strats:
        print(f"  • {s}")

print("\n" + "=" * 70)

       RISK MITIGATION STRATEGIES

Market / Price Risk:
  • Diversification: Spread investments across asset classes (equity, debt, gold) to reduce single-market exposure.
  • Position Sizing: Limit the proportion of portfolio allocated to any single index or sector.
  • Exposure Limits: Set maximum allocation thresholds for equity exposure based on risk tolerance.
  • Long-Term Horizon: Maintain a long investment horizon to ride through short-term market declines.

Volatility Risk:
  • Volatility Monitoring: Track rolling volatility metrics regularly to identify unusual market conditions.
  • Risk Budgets: Allocate risk budgets across portfolio components; reduce exposure when volatility exceeds thresholds.
  • Adaptive Allocation: Consider adjusting portfolio allocation during periods of elevated volatility.
  • Options Hedging: Use protective strategies to limit downside during volatile periods (if available).

Extreme Return / Shock Risk:
  • Stress Testing: Regularly test portfoli

## 12. Key Findings

The following summarizes the most important findings from this risk analysis.

In [22]:
# Key findings summary
print("=" * 70)
print("       KEY FINDINGS")
print("=" * 70)

findings = [
    f"1. The NIFTY 50 index has experienced a price range of ₹{min_close:,.0f} to ₹{max_close:,.0f} "
    f"over the analyzed period ({df['Date'].min().date()} to {df['Date'].max().date()}), "
    f"representing a total appreciation of {((max_close/min_close)-1)*100:,.1f}%.",

    f"2. The maximum drawdown observed was {max_dd_pct:.2f}% (₹{abs(max_dd_abs):,.0f} absolute loss), "
    f"which occurred around {max_dd_date}. {'This drawdown has since recovered.' if isinstance(recovery_days, int) else 'This drawdown has not yet fully recovered.'}",

    f"3. Average 30-day rolling volatility was {avg_vol:.2f}%, with a peak of {max_vol:.2f}% "
    f"observed on {max_vol_date}. {len(high_vol_days)} days exceeded the high-volatility threshold.",

    f"4. The worst single-day loss was {max_loss:.2f}% on {max_loss_date}, and the best "
    f"single-day gain was +{max_gain:.2f}% on {max_gain_date}.",

    f"5. There were {len(large_positive) + len(large_negative)} days with moves exceeding ±{threshold}%, "
    f"representing {(len(large_positive) + len(large_negative))/total_days*100:.2f}% of all trading days.",

    f"6. The return distribution has a kurtosis of {daily_ret.kurtosis():.2f}, indicating "
    f"significantly fat tails compared to a normal distribution (which has kurtosis = 0).",

    f"7. Historical VaR at 95% confidence is {var_95:.2f}% daily, meaning there is a 5% chance "
    f"of losing more than {abs(var_95):.2f}% in a single day.",

    f"8. Negative trading days occurred on {neg_days/total_days*100:.1f}% of all trading days, "
    f"with an average negative return of {negative_returns.mean():.4f}%.",

    f"9. The year {yearly_vol.index[0]} recorded the highest average volatility at {yearly_vol.iloc[0]:.4f}%, "
    f"while {yearly_vol.index[-1]} was the least volatile at {yearly_vol.iloc[-1]:.4f}%.",

    f"10. The most severe risk categories are Market/Price Risk and Drawdown Risk (both rated Very High), "
    f"followed by Volatility Risk and Extreme Return Risk (both rated High)."
]

for f in findings:
    print(f"\n{f}")

print("\n" + "=" * 70)

       KEY FINDINGS

1. The NIFTY 50 index has experienced a price range of ₹2,524 to ₹26,329 over the analyzed period (2007-09-17 to 2026-02-20), representing a total appreciation of 943.0%.

2. The maximum drawdown observed was -59.86% (₹3,764 absolute loss), which occurred around 2008-10-27. This drawdown has since recovered.

3. Average 30-day rolling volatility was 1.10%, with a peak of 4.84% observed on 2008-11-24. 227 days exceeded the high-volatility threshold.

4. The worst single-day loss was -12.98% on 2020-03-23, and the best single-day gain was +17.74% on 2009-05-18.

5. There were 387 days with moves exceeding ±2.0%, representing 8.56% of all trading days.

6. The return distribution has a kurtosis of 16.07, indicating significantly fat tails compared to a normal distribution (which has kurtosis = 0).

7. Historical VaR at 95% confidence is -1.83% daily, meaning there is a 5% chance of losing more than 1.83% in a single day.

8. Negative trading days occurred on 46.8% of 

## 13. Limitations

It is important to acknowledge the limitations of this risk analysis:

1. **Historical data cannot capture every future market event.** Past risk measures may not fully predict future risk.
2. **External macroeconomic and geopolitical factors are not fully represented.** The analysis relies solely on price and return data.
3. **Historical risk measures do not guarantee future outcomes.** Risk levels can change significantly.
4. **Index-level analysis does not represent the risk of every individual constituent.** The NIFTY 50 is a composite index.
5. **Risk classifications are analytical judgments** based on the selected methodology and thresholds.
6. **The analysis is educational and not financial advice.** No investment decisions should be based solely on this analysis.
7. **Volume data quality issues:** The dataset contains zero volume values for many earlier observations, limiting volume-based risk analysis.
8. **Survivorship bias:** The NIFTY 50 constituents change over time; the index only includes current constituents.

## 14. Conclusion

### Summary

This Week 3 risk analysis examined five key risk dimensions of the NIFTY 50 index using historical data from September 2007 to February 2026.

**Main Risks Identified:**
- **Drawdown Risk (Very High):** Peak-to-trough declines can be substantial, requiring long recovery periods.
- **Volatility Risk (High):** Rolling volatility varies significantly across time, with periods of elevated uncertainty.
- **Extreme Return Risk (High):** The index has experienced single-day moves exceeding ±12%, with fat-tailed return distribution.

**Key Mitigation Approaches:**
- Diversification across asset classes
- Position sizing and exposure limits
- Volatility monitoring and adaptive allocation
- Stress testing and scenario analysis
- Maximum drawdown limits and rebalancing policies

**Important Note:** This analysis is based on historical data and should be used for educational purposes. Risk measures are estimates based on past behavior and do not guarantee future outcomes. Continuous risk monitoring is essential for effective risk management.